# Plantilla MatIA · Videos matemáticos con Manim y voz (Google Colab gratuito)
**Momento 4 — Animación y video matemático · MATEM-DOCENTES 2026 · Red Académica MatIA**

Este cuaderno produce videos narrados (60–120 s) con **el mismo formato** siempre: misma voz, misma paleta, misma portada y cierre. Lo fijo vive en `estilo_matia.py` (sección 3, **no se edita**). Cada video nuevo es solo un archivo `video_xxx.py` (sección 4) que la IA genera con el prompt de Seis R.

**Primera vez:** `Entorno de ejecución → Ejecutar todo`, autorizar Drive, esperar 6–10 min.
**Otro video:** ejecutar solo las secciones 4, 5 y 6 con el archivo nuevo.

## 1. Instalación (se repite en cada sesión nueva de Colab; 5–8 min)

In [ ]:
%%capture
!apt-get -qq update
!apt-get -qq install -y libcairo2-dev libpango1.0-dev ffmpeg sox \
    texlive texlive-latex-extra texlive-fonts-recommended texlive-science dvisvgm
!pip install -q manim "manim-voiceover[gtts]"


In [ ]:
import manim, manim_voiceover
print("Manim", manim.__version__, "| manim-voiceover", manim_voiceover.__version__)

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Manim 0.22.1 | manim-voiceover 0.4.0


## 2. Google Drive: carpeta permanente de la serie

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
CARPETA = '/content/drive/MyDrive/Videos_MatIA'
os.makedirs(CARPETA, exist_ok=True)
# Si ya existe caché de voz de sesiones anteriores, restaurarla (evita volver a llamar a gTTS)
if os.path.isdir(f"{CARPETA}/voiceovers"):
    os.makedirs('media', exist_ok=True)
    shutil.copytree(f"{CARPETA}/voiceovers", 'media/voiceovers', dirs_exist_ok=True)
print("Carpeta de la serie:", CARPETA)

Mounted at /content/drive
Carpeta de la serie: /content/drive/MyDrive/Videos_MatIA


## 3. Identidad fija de la serie (`estilo_matia.py`) — NO editar por video
Si algún día querés cambiar la voz (acento) o la paleta para **toda** la serie, se cambia aquí una sola vez y se vuelven a renderizar los videos.

In [ ]:
%%writefile estilo_matia.py
"""
estilo_matia.py — Identidad fija de los videos MatIA (NO editar para cada video).

Todo video hereda de VideoMatIA. Así, voz, colores, portada, cierre y ejes
son idénticos en todos los videos; cada escena nueva solo aporta contenido.

API disponible para las escenas (y para el prompt que se da a la IA):
    self.narrar(texto)                    -> context manager: anima mientras se oye la voz
    self.portada(titulo, subtitulo="")    -> portada estándar (ya narra el título)
    self.cierre(lineas)                   -> resumen final estándar (lista de str en Tex)
    self.ejes(x_range, y_range, ...)      -> Axes con el estilo de la serie
    self.formula(tex, color=None)         -> MathTex en esquina superior izquierda
    self.panel(lineas)                    -> columna de Tex alineada a la izquierda
    self.limpiar()                        -> desvanece todo lo que esté en pantalla
    Colores: C_FUNCION, C_FUNCION2, C_FUNCION3, C_ENFASIS, C_OK, C_ALERTA
"""

from manim import *
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.gtts import GTTSService

# ---------- Paleta fija de la serie ----------
C_FONDO = "#0f1b2d"
C_TEXTO = "#f4f4f4"
C_FUNCION = "#4cc9f0"     # azul claro
C_FUNCION2 = "#f72585"    # magenta
C_FUNCION3 = "#b5e48c"    # verde claro
C_ENFASIS = "#ffd166"     # amarillo
C_OK = "#80ed99"
C_ALERTA = "#ff6b6b"

VOZ_IDIOMA = "es"
VOZ_ACENTO = "com"        # "com" neutro | "es" España | "com.mx" México
SERIE = "MatIA · Red Académica MatIA"
INSTITUCION = "MATEM-UNA · Precálculo 2026"


class VideoMatIA(VoiceoverScene):
    """Escena base: fija voz, fondo, tipografía y bloques reutilizables."""

    def setup(self):
        super().setup()
        self.camera.background_color = C_FONDO
        Text.set_default(color=C_TEXTO)
        Tex.set_default(color=C_TEXTO)
        MathTex.set_default(color=C_TEXTO)
        self.set_speech_service(GTTSService(lang=VOZ_IDIOMA, tld=VOZ_ACENTO))

    # ----- voz -----
    def narrar(self, texto):
        return self.voiceover(text=texto)

    # ----- bloques estándar -----
    def portada(self, titulo, subtitulo=""):
        t = Text(titulo, font_size=44, weight=BOLD)
        s = Text(subtitulo, font_size=26, color=C_ENFASIS) if subtitulo else VGroup()
        pie = Text(f"{SERIE}  ·  {INSTITUCION}", font_size=18, color=GREY_B).to_edge(DOWN)
        grupo = VGroup(t, s).arrange(DOWN, buff=0.4)
        with self.narrar(f"{titulo}. {subtitulo}" if subtitulo else titulo):
            self.play(Write(t), FadeIn(pie))
            if subtitulo:
                self.play(FadeIn(s, shift=UP))
        self.play(FadeOut(s), FadeOut(pie), t.animate.scale(0.6).to_edge(UP, buff=0.3))
        self.titulo_actual = t

    def cierre(self, lineas, narracion=None):
        self.limpiar()
        items = [Tex(r"\textbf{Resumen}", font_size=40, color=C_ENFASIS)]
        items += [Tex(f"{i+1}. {l}", font_size=32) for i, l in enumerate(lineas)]
        g = VGroup(*items).arrange(DOWN, aligned_edge=LEFT, buff=0.4)
        if g.width > 12:
            g.scale_to_fit_width(12)
        texto = narracion or " ".join(lineas)
        with self.narrar(texto):
            self.play(FadeIn(g, lag_ratio=0.3))
        pie = Text(SERIE, font_size=18, color=GREY_B).to_edge(DOWN)
        self.play(FadeIn(pie))
        self.wait(2)

    def ejes(self, x_range, y_range, x_length=7, y_length=4.5, **kw):
        ax = Axes(
            x_range=x_range, y_range=y_range,
            x_length=x_length, y_length=y_length, tips=False,
            axis_config={"include_numbers": True, "font_size": 24, "color": GREY_B},
            **kw,
        ).to_edge(DOWN, buff=0.4)
        return ax

    def formula(self, tex, color=None, font_size=36):
        m = MathTex(tex, font_size=font_size, color=color or C_TEXTO)
        m.to_corner(UL).shift(DOWN * 0.9)
        return m

    def panel(self, lineas, font_size=32, debajo_de=None):
        g = VGroup(*[Tex(l, font_size=font_size) if not isinstance(l, Mobject) else l
                     for l in lineas]).arrange(DOWN, aligned_edge=LEFT, buff=0.25)
        if debajo_de is not None:
            g.next_to(debajo_de, DOWN, aligned_edge=LEFT, buff=0.3)
        else:
            g.to_corner(UL).shift(DOWN * 0.9)
        return g

    def limpiar(self):
        conservar = getattr(self, "titulo_actual", None)
        objetos = [m for m in self.mobjects if m is not conservar]
        if objetos:
            self.play(*[FadeOut(m) for m in objetos])


Writing estilo_matia.py


## 4. La escena del video (archivo generado con el prompt de Seis R)
Pegá aquí el código que te devolvió la IA. Reglas: primera línea `%%writefile video_NOMBRE.py`; sin las marcas ```python. El ejemplo cargado es el **Modelo D** de la serie (bloque II: inversa como reflexión respecto a y = x). En 4b hay un segundo ejemplo (puntos críticos, tema de Cálculo, solo demostración técnica).

In [ ]:
%%writefile video_inversa_reflexion.py
"""
video_inversa_reflexion.py — MODELO D de la serie MatIA (Momento 4, MATEM-Precálculo 2026)
Bloque II · Funciones / exponencial y logarítmica.

APRENDIZAJE: la gráfica de la función inversa es la reflexión de la gráfica de f
respecto a la recta y = x, porque cada par (a, b) de f se convierte en (b, a) de f⁻¹;
esa reflexión solo es una función cuando f es inyectiva.

SOLUCIÓN DE REFERENCIA (resuelta antes de pedir el código):
  f(x) = 2x + 1  → f⁻¹(x) = (x − 1)/2.  Pares: (0,1)→(1,0); (1,3)→(3,1); (2,5)→(5,2).
  g(x) = x²      → su reflejo x = y² NO es función (a x = 4 le corresponden y = 2 y y = −2).
                   Restringida a x ≥ 0, la inversa es √x: (0,0)→(0,0); (1,1)→(1,1); (2,4)→(4,2).
  h(x) = 2ˣ      → h⁻¹(x) = log₂ x.  Pares: (0,1)→(1,0); (1,2)→(2,1); (2,4)→(4,2).
  En los tres casos los puntos de la gráfica sobre y = x (si los hay) no se mueven.
"""
from manim import *
from estilo_matia import *


class InversaReflexion(VideoMatIA):
    def construct(self):
        # ---------- ESCENA 1 · Portada ----------
        self.portada("La función inversa como reflexión",
                     "¿Qué pasa con cada punto al invertir una función?")

        # ---------- ESCENA 2 · Pregunta de predicción y definición ----------
        ax = self.ejes([-1, 5, 1], [-1, 5, 1], x_length=5.5, y_length=5.5).shift(RIGHT * 2.2)
        diag = ax.plot(lambda x: x, x_range=[-1, 5], color=GREY_B, stroke_width=2)
        etq_diag = MathTex("y=x", font_size=28, color=GREY_B).next_to(ax.c2p(4.6, 4.6), RIGHT, buff=0.1)
        p13 = Dot(ax.c2p(1, 3), color=C_FUNCION)
        e13 = MathTex("(1,\\,3)", font_size=30, color=C_FUNCION).next_to(p13, UL, buff=0.1)
        pregunta = self.panel([Tex(r"\textbf{Predecí:}", font_size=30, color=C_ENFASIS),
                               Tex(r"si $(1,3)$ es un punto de $f$,", font_size=28),
                               Tex(r"¿qué punto tiene $f^{-1}$?", font_size=28)])
        with self.narrar("Antes de ver la respuesta, pensá: si el punto uno coma tres está en la "
                         "gráfica de una función f, ¿qué punto debe estar en la gráfica de su inversa?"):
            self.play(Create(ax), Create(diag), Write(etq_diag))
            self.play(FadeIn(p13), Write(e13), Write(pregunta))
        p31 = Dot(ax.c2p(3, 1), color=C_FUNCION2)
        e31 = MathTex("(3,\\,1)", font_size=30, color=C_FUNCION2).next_to(p31, DR, buff=0.1)
        seg = DashedLine(ax.c2p(1, 3), ax.c2p(3, 1), color=C_ENFASIS)
        defin = self.panel([MathTex(r"(a,b)\in f \iff (b,a)\in f^{-1}", font_size=32, color=C_ENFASIS),
                            Tex(r"Intercambiar coordenadas es", font_size=26),
                            Tex(r"reflejar respecto a $y=x$.", font_size=26)], debajo_de=pregunta)
        with self.narrar("La inversa deshace lo que hace f: si f manda uno a tres, la inversa manda "
                         "tres a uno. El punto uno coma tres se convierte en tres coma uno. "
                         "Intercambiar las coordenadas es exactamente reflejar el punto respecto a "
                         "la recta y igual a x: la recta es el eje de simetría."):
            self.play(Create(seg))
            self.play(TransformFromCopy(p13, p31), Write(e31))
            self.play(Write(defin))
        self.limpiar()

        # ---------- ESCENA 3 · Ejemplo lineal f(x)=2x+1 ----------
        ax = self.ejes([-1, 5, 1], [-1, 5, 1], x_length=5.5, y_length=5.5).shift(RIGHT * 2.2)
        diag = ax.plot(lambda x: x, x_range=[-1, 5], color=GREY_B, stroke_width=2)
        f = lambda x: 2 * x + 1
        gf = ax.plot(f, x_range=[-1, 2], color=C_FUNCION)
        ef = self.formula(r"f(x)=2x+1", color=C_FUNCION)
        pts = VGroup(*[Dot(ax.c2p(a, f(a)), color=C_FUNCION) for a in (0, 1, 2)])
        with self.narrar("Primer ejemplo: f de x igual a dos x más uno. Marcamos tres puntos: "
                         "cero coma uno, uno coma tres y dos coma cinco."):
            self.play(Create(ax), Create(diag), Create(gf), Write(ef))
            self.play(FadeIn(pts))
        pts_inv = VGroup(*[Dot(ax.c2p(f(a), a), color=C_FUNCION2) for a in (0, 1, 2)])
        finv = lambda x: (x - 1) / 2
        ginv = ax.plot(finv, x_range=[-1, 5], color=C_FUNCION2)
        einv = MathTex(r"f^{-1}(x)=\frac{x-1}{2}", font_size=36, color=C_FUNCION2).next_to(ef, DOWN, aligned_edge=LEFT, buff=0.4)
        with self.narrar("Reflejamos cada punto: cero coma uno pasa a uno coma cero; uno coma tres "
                         "a tres coma uno; dos coma cinco a cinco coma dos. Los puntos reflejados "
                         "quedan sobre una nueva recta."):
            self.play(TransformFromCopy(pts, pts_inv))
        with self.narrar("Esa recta es la gráfica de la inversa: f inversa de x igual a x menos uno "
                         "sobre dos. Comprobá con un punto: f inversa de tres es tres menos uno "
                         "sobre dos, igual a uno. Correcto."):
            self.play(TransformFromCopy(gf, ginv), Write(einv))
        self.limpiar()

        # ---------- ESCENA 4 · Contraejemplo g(x)=x² ----------
        ax = self.ejes([-2.5, 5, 1], [-2.5, 5, 1], x_length=5.5, y_length=5.5).shift(RIGHT * 2.2)
        diag = ax.plot(lambda x: x, x_range=[-2.5, 5], color=GREY_B, stroke_width=2)
        g = lambda x: x ** 2
        gg = ax.plot(g, x_range=[-2.2, 2.2], color=C_FUNCION)
        eg = self.formula(r"g(x)=x^2", color=C_FUNCION)
        with self.narrar("Segundo ejemplo, el que rompe la idea ingenua: g de x igual a x cuadrado. "
                         "¿Basta con reflejar la parábola para obtener la inversa?"):
            self.play(Create(ax), Create(diag), Create(gg), Write(eg))
        refl = ax.plot_parametric_curve(lambda t: np.array([t ** 2, t, 0]), t_range=[-2.2, 2.2], color=C_ALERTA)
        with self.narrar("Reflejamos. La curva que resulta es x igual a y cuadrado."):
            self.play(TransformFromCopy(gg, refl))
        vert = DashedLine(ax.c2p(4, -2.5), ax.c2p(4, 5), color=C_ENFASIS)
        d1, d2 = Dot(ax.c2p(4, 2), color=C_ENFASIS), Dot(ax.c2p(4, -2), color=C_ENFASIS)
        aviso = self.panel([Tex(r"A $x=4$ le corresponden", font_size=28),
                            MathTex(r"y=2\ \text{ y }\ y=-2", font_size=32, color=C_ALERTA),
                            Tex(r"$\Rightarrow$ el reflejo \textbf{no} es función.", font_size=28),
                            Tex(r"$g$ no es inyectiva: $g(2)=g(-2)=4$.", font_size=26)], debajo_de=eg)
        with self.narrar("Pero fijate: a x igual a cuatro le corresponden dos valores, y igual a dos "
                         "y y igual a menos dos. El reflejo no es una función. La razón es que g no "
                         "es inyectiva: dos y menos dos tienen la misma imagen, cuatro."):
            self.play(Create(vert), FadeIn(d1), FadeIn(d2))
            self.play(Write(aviso))
        rama = ax.plot(g, x_range=[0, 2.2], color=C_FUNCION3, stroke_width=6)
        raiz = ax.plot(lambda x: np.sqrt(x), x_range=[0, 5], color=C_FUNCION3, stroke_width=6)
        eraiz = MathTex(r"x\ge 0:\quad g^{-1}(x)=\sqrt{x}", font_size=32, color=C_FUNCION3).next_to(aviso, DOWN, aligned_edge=LEFT, buff=0.3)
        with self.narrar("Si restringimos el dominio a x mayor o igual que cero, g sí es inyectiva, "
                         "y el reflejo de esa rama es la función raíz cuadrada de x. El punto dos "
                         "coma cuatro pasa a cuatro coma dos."):
            self.play(FadeOut(refl), FadeOut(vert), FadeOut(d1), FadeOut(d2), Create(rama))
            self.play(TransformFromCopy(rama, raiz), Write(eraiz))
        self.limpiar()

        # ---------- ESCENA 5 · h(x)=2^x y log2 ----------
        ax = self.ejes([-1.5, 5, 1], [-1.5, 5, 1], x_length=5.5, y_length=5.5).shift(RIGHT * 2.2)
        diag = ax.plot(lambda x: x, x_range=[-1.5, 5], color=GREY_B, stroke_width=2)
        h = lambda x: 2 ** x
        gh = ax.plot(h, x_range=[-1.5, 2.3], color=C_FUNCION)
        eh = self.formula(r"h(x)=2^x", color=C_FUNCION)
        pts = VGroup(*[Dot(ax.c2p(a, h(a)), color=C_FUNCION) for a in (0, 1, 2)])
        with self.narrar("Tercer ejemplo: la exponencial h de x igual a dos elevado a la x, con los "
                         "puntos cero coma uno, uno coma dos y dos coma cuatro. Es creciente, así que "
                         "es inyectiva y tiene inversa en todo su dominio."):
            self.play(Create(ax), Create(diag), Create(gh), Write(eh), FadeIn(pts))
        pts_inv = VGroup(*[Dot(ax.c2p(h(a), a), color=C_FUNCION2) for a in (0, 1, 2)])
        glog = ax.plot(lambda x: np.log2(x), x_range=[0.35, 5], color=C_FUNCION2)
        elog = MathTex(r"h^{-1}(x)=\log_2 x", font_size=36, color=C_FUNCION2).next_to(eh, DOWN, aligned_edge=LEFT, buff=0.4)
        nota = self.panel([Tex(r"$\log_2 4 = 2$ porque $2^2=4$", font_size=28),
                           Tex(r"Dominio de $\log_2 x$: $x>0$", font_size=28, color=C_ENFASIS)], debajo_de=elog)
        with self.narrar("Reflejamos: uno coma cero, dos coma uno, cuatro coma dos. La curva reflejada "
                         "es el logaritmo en base dos. Logaritmo de cuatro es dos porque dos al "
                         "cuadrado es cuatro: la inversa responde la pregunta «¿a qué exponente?»."):
            self.play(TransformFromCopy(pts, pts_inv))
            self.play(TransformFromCopy(gh, glog), Write(elog))
        with self.narrar("Y observá el dominio: la exponencial solo toma valores positivos, así que su "
                         "inversa solo acepta entradas positivas. Por eso el logaritmo existe únicamente "
                         "para x mayor que cero."):
            self.play(Write(nota))

        # ---------- ESCENA 6 · Cierre ----------
        self.cierre(
            [r"$(a,b)\in f \iff (b,a)\in f^{-1}$: la inversa refleja respecto a $y=x$.",
             r"El reflejo es función solo si $f$ es inyectiva ($x^2$ necesita $x\ge 0$).",
             r"$2^x$ y $\log_2 x$ son reflejo una de la otra; dominio de $\log_2 x$: $x>0$.",
             r"Para comprobar: $f^{-1}(f(a))=a$ con un punto concreto."],
            narracion="Resumiendo. La gráfica de la inversa es la reflexión de la gráfica de f "
                      "respecto a la recta y igual a x, porque cada par a coma b se convierte en b "
                      "coma a. Ese reflejo es una función solo cuando f es inyectiva; por eso x "
                      "cuadrado necesita restringirse. La exponencial y el logaritmo son reflejo una "
                      "de la otra, y el dominio del logaritmo hereda el ámbito de la exponencial. "
                      "Siempre comprobá con un punto concreto.",
        )


# =============================================================================
# STORYBOARD (salida de la R de Revisión) — una fila por escena
# | Esc | Estado de pantalla                     | Qué se mueve / aparece                    | Valor de control y justificación                          |
# |  1  | Portada + subtítulo                    | Título                                    | —                                                          |
# |  2  | Ejes cuadrados, y=x, punto (1,3)       | Pregunta de predicción; reflejo a (3,1)   | (1,3)→(3,1): intercambio de coordenadas                    |
# |  3  | f(x)=2x+1 con (0,1),(1,3),(2,5)        | Puntos reflejados; recta f⁻¹=(x−1)/2       | f⁻¹(3)=(3−1)/2=1 ✔; (2,5)→(5,2) sobre la recta              |
# |  4  | g(x)=x² y su reflejo x=y²              | Recta x=4 corta en y=2 y y=−2; rama x≥0→√x | g(2)=g(−2)=4 (no inyectiva); √4=2 ✔                        |
# |  5  | h(x)=2ˣ con (0,1),(1,2),(2,4)          | Puntos reflejados; curva log₂x            | log₂4=2 porque 2²=4 ✔; log₂x definido solo si x>0          |
# |  6  | Resumen de 4 líneas                    | Aparición escalonada                      | —                                                          |
# Duración estimada: 105–120 s (13 bloques de narración). Render de prueba: manim -ql video_inversa_reflexion.py InversaReflexion
# =============================================================================


Writing video_inversa_reflexion.py


### 4b. Segundo ejemplo (opcional): puntos críticos
Ejecutá esta celda solo si querés renderizar también este video; luego en la sección 5 usá `ARCHIVO = "video_puntos_criticos.py"`, `CLASE = "PuntosCriticos"`.

In [ ]:
%%writefile video_puntos_criticos.py
"""
video_puntos_criticos.py — Ejemplo modelo de la serie MatIA.
Aprendizaje: un punto crítico (f'(c)=0 o f'(c) no existe) es candidato a extremo,
no necesariamente máximo ni mínimo.
"""
from manim import *
from estilo_matia import *


class PuntosCriticos(VideoMatIA):
    def construct(self):
        self.portada("Puntos críticos de una función",
                     "¿Siempre son máximos o mínimos?")

        # ---------- Definición ----------
        defin = VGroup(
            Tex(r"$c$ es \textbf{punto crítico} de $f$ si", font_size=36),
            MathTex(r"f'(c)=0 \qquad \text{o bien} \qquad f'(c)\ \text{no existe}",
                    font_size=40, color=C_ENFASIS),
            Tex(r"\textbf{Fermat:} si $f$ tiene extremo relativo en $c$ y $f'(c)$ existe, "
                r"entonces $f'(c)=0$.", font_size=30),
        ).arrange(DOWN, buff=0.5)
        with self.narrar("Decimos que c es un punto crítico de f si la derivada en c es "
                         "cero, o bien si la derivada en c no existe. La definición solo "
                         "habla de la derivada; no dice que ahí haya un máximo o un mínimo."):
            self.play(FadeIn(defin[0], shift=UP), Write(defin[1]))
        with self.narrar("Por el teorema de Fermat, todo extremo relativo ocurre en un punto "
                         "crítico. Ser punto crítico es condición necesaria, no suficiente: "
                         "los puntos críticos son candidatos que hay que verificar."):
            self.play(FadeIn(defin[2]))
        self.limpiar()

        # ---------- Ejemplo 1: f(x)=x^3-3x ----------
        ax = self.ejes([-3, 3, 1], [-4, 4, 1])
        f = lambda x: x**3 - 3 * x
        gf = ax.plot(f, x_range=[-2.3, 2.3], color=C_FUNCION)
        e1 = self.formula(r"f(x)=x^3-3x", color=C_FUNCION)
        p1 = self.panel([MathTex(r"f'(x)=3x^2-3=3(x-1)(x+1)", font_size=32),
                         MathTex(r"f'(x)=0 \iff x=-1 \ \text{o}\ x=1", font_size=32)],
                        debajo_de=e1)
        with self.narrar("Primer ejemplo: f de x igual a x al cubo menos tres x."):
            self.play(Create(ax), Create(gf), Write(e1))
        with self.narrar("Su derivada, tres x cuadrado menos tres, existe en todo real y se "
                         "anula en x igual a menos uno y en x igual a uno. Esos son los "
                         "puntos críticos."):
            self.play(Write(p1))
        pmax, pmin = Dot(ax.c2p(-1, f(-1)), color=C_ENFASIS), Dot(ax.c2p(1, f(1)), color=C_ENFASIS)
        tmax = DashedLine(ax.c2p(-2, f(-1)), ax.c2p(0, f(-1)), color=C_ENFASIS)
        tmin = DashedLine(ax.c2p(0, f(1)), ax.c2p(2, f(1)), color=C_ENFASIS)
        lmax = Tex("máximo relativo", font_size=26, color=C_OK).next_to(pmax, UP)
        lmin = Tex("mínimo relativo", font_size=26, color=C_OK).next_to(pmin, DOWN)
        with self.narrar("En ambos la tangente es horizontal. Al revisar el signo de la derivada "
                         "a cada lado, en menos uno hay un máximo relativo y en uno un mínimo "
                         "relativo. Aquí los críticos sí fueron extremos. No siempre pasa."):
            self.play(FadeIn(pmax), FadeIn(pmin), Create(tmax), Create(tmin))
            self.play(Write(lmax), Write(lmin))
        self.limpiar()

        # ---------- Ejemplo 2: g(x)=x^3 ----------
        ax = self.ejes([-2, 2, 1], [-4, 4, 2])
        g = lambda x: x**3
        gg = ax.plot(g, x_range=[-1.55, 1.55], color=C_FUNCION2)
        e2 = self.formula(r"g(x)=x^3", color=C_FUNCION2)
        p2 = self.panel([MathTex(r"g'(x)=3x^2,\quad g'(0)=0", font_size=32),
                         MathTex(r"g'(x)>0\ \ \forall x\neq 0", font_size=32)], debajo_de=e2)
        with self.narrar("Segundo ejemplo, el más importante: g de x igual a x al cubo. "
                         "Su derivada, tres x cuadrado, vale cero en x igual a cero, así que "
                         "cero es punto crítico."):
            self.play(Create(ax), Create(gg), Write(e2), Write(p2[0]))
        p0 = Dot(ax.c2p(0, 0), color=C_ENFASIS)
        t0 = DashedLine(ax.c2p(-1, 0), ax.c2p(1, 0), color=C_ENFASIS)
        c2 = Tex(r"$x=0$ es crítico pero \textbf{no} es máximo ni mínimo",
                 font_size=30, color=C_ALERTA).next_to(ax, UP, buff=0.05).shift(RIGHT * 1.5)
        with self.narrar("La tangente en el origen es horizontal, pero la función nunca deja "
                         "de crecer: la derivada es positiva para todo x distinto de cero y no "
                         "cambia de signo. Conclusión: cero es punto crítico, pero no es máximo "
                         "ni mínimo. Es un punto de inflexión con tangente horizontal."):
            self.play(FadeIn(p0), Create(t0), Write(p2[1]))
            self.play(Write(c2))
        self.limpiar()

        # ---------- Ejemplo 3: h(x)=x^(2/3) ----------
        ax = self.ejes([-2, 2, 1], [-0.5, 2, 0.5])
        h = lambda x: np.cbrt(x) ** 2
        gh = VGroup(ax.plot(h, x_range=[-2, 0], color=C_FUNCION3),
                    ax.plot(h, x_range=[0, 2], color=C_FUNCION3))
        e3 = self.formula(r"h(x)=x^{2/3}", color=C_FUNCION3)
        p3 = self.panel([MathTex(r"h'(x)=\frac{2}{3\sqrt[3]{x}},\quad h'(0)\ \text{no existe}",
                                 font_size=32)], debajo_de=e3)
        pc = Dot(ax.c2p(0, 0), color=C_ENFASIS)
        c3 = Tex(r"$x=0$ es crítico porque $h'(0)$ no existe; aquí sí hay mínimo",
                 font_size=30, color=C_OK).next_to(ax, UP, buff=0.05).shift(RIGHT * 1.2)
        with self.narrar("Tercer ejemplo: h de x igual a x elevado a dos tercios. Su derivada "
                         "no está definida en cero, así que cero es punto crítico por la "
                         "segunda parte de la definición: no hay tangente horizontal, hay una "
                         "cúspide."):
            self.play(Create(ax), Create(gh), Write(e3), Write(p3))
        with self.narrar("En este caso sí hay un mínimo, porque h es mayor o igual que cero y "
                         "vale cero en el origen. Pero lo que lo hace crítico es que la derivada "
                         "no existe, no que sea mínimo."):
            self.play(FadeIn(pc), Write(c3))

        # ---------- Cierre ----------
        self.cierre(
            [r"Punto crítico: $f'(c)=0$ o $f'(c)$ no existe.",
             r"Todo extremo relativo está en un punto crítico (Fermat).",
             r"No todo punto crítico es extremo: $g(x)=x^3$ en $x=0$.",
             r"Para decidir: criterio de la primera o segunda derivada."],
            narracion="Resumiendo. Punto crítico: derivada cero o inexistente. Todo extremo "
                      "relativo está en un punto crítico, pero no todo punto crítico es extremo; "
                      "x al cubo en cero es el contraejemplo. Para decidir, usamos el criterio de "
                      "la primera o de la segunda derivada. Los puntos críticos son candidatos, "
                      "no conclusiones.",
        )


Writing video_puntos_criticos.py


## 5. Renderizar
Cambiá `ARCHIVO` y `CLASE` por los del video nuevo. `-ql` = prueba rápida (480p); `-qh` = final (1080p).

In [ ]:
ARCHIVO = "video_inversa_reflexion.py"
CLASE   = "InversaReflexion"
CALIDAD = "-ql"      # -ql prueba | -qm 720p | -qh 1080p
!manim {CALIDAD} {ARCHIVO} {CLASE}

Manim Community v0.22.1

[11/03/26 21:12:06] INFO     Automatically converted    ]8;id=944540;file:///usr/local/lib/python3.13/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=510363;file:///usr/local/lib/python3.13/dist-packages/manim/scene/scene_file_writer.py#459\459]8;;\
                             media/voiceovers/la-funcio                         
                             n-inversa-como-reflexion-q                         
                             ue-pasa-con-a5060102.mp3                           
                             to .wav                                            
[11/03/26 21:12:07] INFO     Animation 0 : Partial      ]8;id=613047;file:///usr/local/lib/python3.13/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=549891;file:///usr/local/lib/python3.13/dist-packages/manim/scene/scene_file_writer.py#173\173]8;;\
                             movie file written in                       

## 6. Ver, guardar en Drive y conservar la caché de voz

In [ ]:
from IPython.display import Video
import glob, datetime
base = ARCHIVO.replace('.py','')
ruta = sorted(glob.glob(f'media/videos/{base}/*/{CLASE}.mp4'), key=os.path.getmtime)[-1]
destino = f"{CARPETA}/{CLASE}_{datetime.date.today()}.mp4"
shutil.copy(ruta, destino)
shutil.copy(ARCHIVO, f"{CARPETA}/{ARCHIVO}")
shutil.copy('estilo_matia.py', f"{CARPETA}/estilo_matia.py")
if os.path.isdir('media/voiceovers'):
    shutil.copytree('media/voiceovers', f"{CARPETA}/voiceovers", dirs_exist_ok=True)
print("Video guardado en:", destino)
Video(ruta, embed=True, width=720)

Video guardado en: /content/drive/MyDrive/Videos_MatIA/InversaReflexion_2026-11-03.mp4


## 6b. Música de fondo (opcional, misma pista para toda la serie)
Subí **una** pista a `MyDrive/Videos_MatIA/musica_serie.mp3` y ejecutá esta celda después de la 6. Usá solo música con licencia libre (CC0 o Creative Commons que permita uso educativo, p. ej. Free Music Archive, Pixabay Music, YouTube Audio Library) o de autoría propia, y declará la fuente en la descripción del video. La música se baja a −22 dB para no competir con la voz; nunca es requisito del entregable.

In [ ]:
MUSICA = f"{CARPETA}/musica_serie.mp3"
VOL_DB = -22            # volumen de la música respecto a la voz
if os.path.exists(MUSICA):
    salida = ruta.replace('.mp4', '_musica.mp4')
    !ffmpeg -y -loglevel error -i "{ruta}" -stream_loop -1 -i "{MUSICA}" \
        -filter_complex "[1:a]volume={VOL_DB}dB[m];[0:a][m]amix=inputs=2:duration=first:dropout_transition=2[a]" \
        -map 0:v -map "[a]" -c:v copy -c:a aac -shortest "{salida}"
    shutil.copy(salida, f"{CARPETA}/{CLASE}_{datetime.date.today()}_musica.mp4")
    print("Con música:", salida); display(Video(salida, embed=True, width=720))
else:
    print("No hay musica_serie.mp3 en la carpeta de Drive; el video queda solo con voz.")

No hay musica_serie.mp3 en la carpeta de Drive; el video queda solo con voz.


## 7. Cómo pedir OTRO video (flujo de Seis R)
1. **Resolvé primero** la matemática del video (valores notables, derivadas, intersecciones). Eso va en la R de *Recursos*.
2. Copiá el prompt de la celda siguiente, completá los corchetes y pegalo en ChatGPT, Claude o Gemini **junto con el contenido de `estilo_matia.py`** (para que la IA conozca la API) y, si querés, `video_puntos_criticos.py` como ejemplo del formato.
3. Pegá la respuesta en una celda nueva con `%%writefile video_NOMBRE.py` (o reemplazá la sección 4).
4. En la sección 5 cambiá `ARCHIVO` y `CLASE`; ejecutá 4 → 5 → 6.
5. **Revisión** (R6): mirá el video completo con la lista de cotejo del manual antes de usarlo en tutoría. Guardá el prompt, la solución de referencia y el `.py` en la carpeta de Drive: esa es la evidencia de trazabilidad del Momento 4.

In [ ]:
PROMPT_6R = r"""ROL
Actuá como especialista en educación matemática de secundaria y programador experto en Manim Community Edition (v0.19 o superior) con manim-voiceover. Trabajás para la serie de videos "MatIA" de MATEM-UNA Precálculo 2026, que ya tiene una identidad visual y de voz definida en el archivo estilo_matia.py (no debés modificarla ni redefinirla).

RESULTADO
Entregá únicamente un archivo Python llamado video_[NOMBRE_CORTO].py que defina UNA clase [NombreClase](VideoMatIA) con el método construct, que produzca un video narrado de entre 60 y 120 segundos sobre:

   [APRENDIZAJE: qué relación matemática debe quedar clara al terminar el video]

Estructura obligatoria en 4 a 6 ESCENAS (cada escena = un estado de pantalla que se limpia antes de la siguiente): self.portada(...) → definición o situación inicial con una PREGUNTA DE PREDICCIÓN antes del primer movimiento → 2 o 3 ejemplos (incluido al menos un contraejemplo o caso que rompa la idea ingenua; un solo parámetro cambia a la vez) → self.cierre([...]). Cada escena lleva de 1 a 3 bloques de narración.

RECEPTOR
Estudiantes de [NIVEL: p. ej. undécimo año / MATEM-Precálculo, bloque II: funciones] que ya conocen [CONOCIMIENTOS PREVIOS] y que suelen cometer el error de [ERROR FRECUENTE QUE EL VIDEO DEBE DESARMAR]. Lenguaje en español de Costa Rica, voseo moderado, sin jerga innecesaria.

RESTRICCIONES
- Empezá con: from manim import * / from estilo_matia import *
- Usá solo la API de estilo_matia: self.narrar(texto) como context manager, self.portada, self.cierre, self.ejes, self.formula, self.panel, self.limpiar, y los colores C_FUNCION, C_FUNCION2, C_FUNCION3, C_ENFASIS, C_OK, C_ALERTA. No crees Axes, colores ni servicios de voz propios.
- Cada idea va dentro de un bloque "with self.narrar(...)": la narración debe decir exactamente lo que se ve, en el mismo orden; nunca anuncies algo que aún no está en pantalla.
- Entre 8 y 14 bloques de narración; cada uno de 1 a 3 oraciones.
- Usá MathTex/Tex para toda expresión matemática; Text solo para títulos.
- Sin archivos externos, sin imágenes, sin bibliotecas adicionales, sin LaTeX exótico (solo amsmath).
- Evitá textos superpuestos: llamá self.limpiar() antes de cambiar de ejemplo; el panel de fórmulas va arriba a la izquierda y los ejes abajo.
- Matemática exacta: los valores que se dibujan (ceros, vértices, puntos marcados) deben coincidir con los que se dicen.
- Entregá solo el código, sin markdown, sin explicación fuera de comentarios.

RECURSOS
- Solución matemática de referencia (resuelta por mí, usala tal cual, no la cambies):
  [PEGAR AQUÍ: funciones, derivadas, valores notables, intervalos, conclusión]
- Programa MATEM-Precálculo 2026, núcleo: [NÚCLEO].
- Ejemplo modelo del formato: video_puntos_criticos.py (misma serie).

REVISIÓN
Antes de entregar, verificá y declará al final del archivo, como comentario, el STORYBOARD: una fila por escena con (1) número y nombre, (2) estado de pantalla, (3) qué se mueve o aparece, (4) texto de la narración, (5) valor de control que debe verificarse cuadro a cuadro y su justificación con la solución de referencia; más la duración estimada total. Si algún valor no coincide con la solución de referencia, corregí el código, no la referencia.
"""
print(PROMPT_6R)

## 8. Plan B si falla `manim-voiceover`
Renderiza mudo, genera un solo mp3 con gTTS y los mezcla con ffmpeg (sincronía aproximada).

In [ ]:
# Plan B — solo si la sección 5 falló por manim-voiceover
import re, subprocess
est = open('estilo_matia.py').read()
est = est.replace("from manim_voiceover import VoiceoverScene\n","").replace("from manim_voiceover.services.gtts import GTTSService\n","")
est = est.replace("class VideoMatIA(VoiceoverScene):","class VideoMatIA(Scene):")
est = est.replace("        self.set_speech_service(GTTSService(lang=VOZ_IDIOMA, tld=VOZ_ACENTO))\n","")
est = est.replace("        return self.voiceover(text=texto)", """        escena = self
        class _Pausa:
            def __enter__(s): s.t0 = escena.renderer.time; return s
            def __exit__(s, *a):
                falta = len(texto)/15.5 - (escena.renderer.time - s.t0)   # ~15 car./s en gTTS
                if falta > 0: escena.wait(falta)
        return _Pausa()""")
open('estilo_matia_mudo.py','w').write(est)
src = open(ARCHIVO).read().replace("from estilo_matia import *","from estilo_matia_mudo import *")
open('mudo_'+ARCHIVO,'w').write(src)
# Guion = portada + narraciones + cierre, en orden de aparición
textos = re.findall(r'(?:narrar\(|portada\(|narracion=)\s*((?:"[^"]*"\s*,?\s*)+)', src)
guion = ". ".join(" ".join(re.findall(r'"([^"]*)"', m)) for m in textos)
from gtts import gTTS; gTTS(guion, lang='es').save('narracion.mp3')
subprocess.run(['manim','-ql','mudo_'+ARCHIVO, CLASE], check=True)
m = sorted(glob.glob(f'media/videos/mudo_{base}/*/{CLASE}.mp4'), key=os.path.getmtime)[-1]
subprocess.run(['ffmpeg','-y','-i',m,'-i','narracion.mp3','-c:v','copy','-c:a','aac','-shortest',f'{CLASE}_planB.mp4'], check=True)
shutil.copy(f'{CLASE}_planB.mp4', CARPETA); print("Listo (Plan B):", f'{CLASE}_planB.mp4')